# RETECO · блок «Запрос» на открытой LLM + BM25 кита

Достраивает варианты запроса, которых не было в прогоне Claude, и расширение с
обратной связью, как у DIVER. Всё ищется **официальным BM25 кита** (gensim +
анализатор Lucene), поэтому числа прямо сравнимы с таблицей: первая ячейка поиска
проверяет, что BM25 + история даёт на dev ровно 0.4379 по доменам.

Варианты (всё приклеивается к «реплика + история», как q2d у Claude):
- `win1`, `win2` — только последние 1–2 пары истории вместо всей (без LLM)
- `stepback` — более общий вопрос над исходным
- `subq` — 2–3 подвопроса; каждым ищется отдельно, списки сливаются RRF
- `re` — последний ответ ассистента переписан самодостаточным
- `q2d_open` — гипотетический абзац открытой модели (тот же q2d, что у Claude)
- `kw` — 30 ключевых слов из этого абзаца, которых нет в запросе
- `qexp1`, `qexp2` — расширение с обратной связью (DIVER QExpand): модель видит
  топ-5 найденного и переписывает абзац; в запросе остаётся реплика + история +
  последнее расширение

**Как запускать.** Среда → **A100** (Qwen3-8B) или **L4** (Qwen3-4B-Instruct-2507).
«Выполнить все». Если ячейка 2 попросит перезапуск — Перезапустить сеанс и снова
«Выполнить все». Оборвалось — ещё раз: ответы LLM и готовые прогоны на Drive.

**Сколько.** ≈3 тыс. реплик dev + train. LLM ≈17 тыс. коротких генераций —
15–30 мин на A100; BM25: индекс 10–15 мин, поиск ещё 10–20.

**Что получится.**
- `reteco/queries/<вариант>/<сплит>.jsonl` — тексты запросов (их потом подхватят
  `diver` и `reason_embed`: там они считаются за минуты)
- `reteco/runs/bm25_<вариант>_<сплит>/` — прогоны
- `reteco/qgen/<модель>/` — сырые ответы LLM (кэш)

**Потом.** Скачать `reteco/runs` и `reteco/queries` в проект, сказать Claude.
Варианты выбираются по train, dev — только отчёт.

In [ ]:
# ── 1. Настройки ─────────────────────────────────────────────────────────────
LLM_MODEL = "auto"    # A100 → Qwen/Qwen3-8B, иначе Qwen/Qwen3-4B-Instruct-2507
SPLITS = ["dev", "train"]
FEEDBACK_K = 5        # сколько найденных документов видит LLM в QExpand
DOC_WORDS = 300       # документ в промпте обрезается до стольких слов
N_KW = 30             # ключевых слов в варианте kw
MAXLEN = 8192
MAX_NEW = 256
TOPK = 100
DRIVE = "/content/drive/MyDrive/reteco"

In [ ]:
# ── 2. Drive, vLLM, GPU ─────────────────────────────────────────────────────
import os, sys, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
from google.colab import drive
drive.mount("/content/drive")
!pip -q install vllm openai pytrec-eval-terrier 2>&1 | tail -2
import torch

# vllm ставит свой torch, а torchvision/torchaudio из Colab остаются под другую
# CUDA, и import vllm падает. Лечение из occ-rag-bfcl: torchaudio снести,
# torchvision поставить под CUDA нового torch. После правки нужен перезапуск.
def _imports_ok(mod):
    try:
        __import__(mod); return True
    except RuntimeError:
        return False       # стоит, но собран под другую CUDA
    except ImportError:
        return True        # не стоит — и конфликта нет
fixed = False
if not _imports_ok("torchaudio"):
    subprocess.run([sys.executable, "-m", "pip", "-q", "uninstall", "-y", "torchaudio"], check=True)
    fixed = True
if not _imports_ok("torchvision") and "+" in torch.__version__:
    base, cu = torch.__version__.split("+")
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "--force-reinstall", "--no-deps",
                    f"torchvision==0.{int(base.split('.')[1]) + 15}.*",
                    "--index-url", f"https://download.pytorch.org/whl/{cu}"], check=True)
    fixed = True
if fixed:
    raise RuntimeError("Зависимости исправлены. Среда выполнения → Перезапустить сеанс, "
                       "потом снова «Выполнить все».")
if globals().get("NEED_GPU", True):
    assert torch.cuda.is_available(), "Нет GPU: Среда выполнения → Сменить среду выполнения → A100 или L4"
MEM = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
DTYPE = "bfloat16" if MEM and torch.cuda.is_bf16_supported() else "half"   # T4 bf16 не умеет
BIG = MEM >= 39                                                             # A100 40/80 ГБ
print(torch.cuda.get_device_name(0) if MEM else "без GPU", f"{MEM:.0f} ГБ, {DTYPE}, torch", torch.__version__)

def load_llm(model, maxlen, util=0.88):
    """На время конструктора — настоящие stdout/stderr: в ядре Jupyter vLLM падает
    на sys.stdout.fileno() ещё до загрузки весов (occ-rag-bfcl)."""
    from vllm import LLM
    out, err = sys.stdout, sys.stderr
    sys.stdout, sys.stderr = sys.__stdout__, sys.__stderr__
    try:
        return LLM(model=model, dtype=DTYPE, max_model_len=maxlen, gpu_memory_utilization=util, seed=0)
    finally:
        sys.stdout, sys.stderr = out, err

def chat(tok, msgs):
    """Промпт по шаблону модели; у Qwen3 размышления выключены (быстро, детерминированно)."""
    return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False, enable_thinking=False)

In [ ]:
# ── 2b. Java 21, pyserini (только анализатор Lucene), gensim, кит организаторов ──
# pyserini ставится без зависимостей: от него нужен только анализатор, а его
# полные зависимости (torch, transformers) сломали бы vLLM.
import glob
!apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-21-jdk-headless > /dev/null
!pip -q install --no-deps pyserini==2.4.0
!pip -q install pyjnius gensim
!git clone -q https://github.com/DataScienceUIBK/RETECO /content/reteco_kit 2>/dev/null || true
jdk = (glob.glob("/usr/lib/jvm/java-21-openjdk*") or [""])[0]
os.environ.setdefault("JAVA_HOME", jdk)
os.environ.setdefault("JVM_PATH", f"{jdk}/lib/server/libjvm.so")
sys.path.insert(0, "/content/reteco_kit/starter_kit")
print("JAVA_HOME", os.environ["JAVA_HOME"])

In [ ]:
# ── 3. Данные RETECO, трек 2 (≈270 МБ, минута) ──────────────────────────────
from huggingface_hub import snapshot_download
snapshot_download("DataScience-UIBK/RETECO-SemEval2027", repo_type="dataset",
                  local_dir="/content/reteco",
                  allow_patterns=["track2_recor/*/documents.jsonl",
                                  "track2_recor/*/benchmark_*.json",
                                  "track2_recor/*/qrels_*.txt"])

In [ ]:
# ── 4. Общий код. Повторяет scripts/retrieve.py и score.py из репозитория ──
import os, re, json, time, glob
import numpy as np, pytrec_eval
from tqdm.auto import tqdm

DATA = "/content/reteco/track2_recor"
DOMAINS = ["drones", "politics", "law", "medicalsciences", "hardware", "economics",
           "psychology", "biology", "sustainable_living", "robotics", "earth_science"]  # от малых к большим
# официальный BM25 «реплика + история», dev, nDCG@10 — для сравнения
BM25_DEV = {"biology": .6065, "drones": .2954, "earth_science": .6264, "economics": .4739,
            "hardware": .3378, "law": .3437, "medicalsciences": .1730, "politics": .3219,
            "psychology": .5726, "robotics": .5123, "sustainable_living": .5531}
BM25_MACRO = {"dev": 0.4379, "train": 0.4539}
RUNS, QUERIES = f"{DRIVE}/runs", f"{DRIVE}/queries"
os.makedirs(RUNS, exist_ok=True); os.makedirs(QUERIES, exist_ok=True)

def load_corpus(dom):
    ids, texts = [], []
    for ln in open(f"{DATA}/{dom}/documents.jsonl", encoding="utf-8"):
        if ln.strip():
            d = json.loads(ln); ids.append(d["doc_id"]); texts.append(d["content"])
    return ids, texts

def topics(dom, split, qset):
    """Реплики без золота пропускаются; история приклеивается как у организаторов.
    qset: "query", "hist" или имя папки в Drive/reteco/queries (варианты rewrite.py)."""
    given = None
    if qset not in ("query", "hist"):
        with open(f"{QUERIES}/{qset}/{split}.jsonl", encoding="utf-8") as f:
            # ключ (домен, topic_id): в доменах BRIGHT topic_id повторяются между доменами
            given = {(r["domain"], r["topic_id"]): r["text"] for r in map(json.loads, filter(str.strip, f))}
    ids, texts = [], []
    for c in json.load(open(f"{DATA}/{dom}/benchmark_{split}.json", encoding="utf-8")):
        for t in c["turns"]:
            if not (t.get("gold_doc_ids") or t.get("supporting_doc_ids")):
                continue
            qid = f"{c['id']}_turn_{t['turn_id']}"
            if given is not None:
                text = given[(dom, qid)]
            else:
                text, h = t["query"], t.get("conversation_history", "")
                if qset == "hist" and h and h != "No previous conversation.":
                    text += f"\n\nConversation History:\n{h}"
            ids.append(qid); texts.append(text)
    return ids, texts

def ndcg10(run, dom, split):
    """run: {qid: {doc: score}}. Реплика без прогона получает 0, как на лидерборде."""
    qrels = {}
    for ln in open(f"{DATA}/{dom}/qrels_{split}.txt"):
        p = ln.split()
        if len(p) == 4:
            qrels.setdefault(p[0], {})[p[2]] = int(p[3])
    res = pytrec_eval.RelevanceEvaluator(qrels, {"ndcg_cut.10"}).evaluate(run)
    return sum(v["ndcg_cut_10"] for v in res.values()) / len(qrels)

def read_run(path):
    run = {}
    for ln in open(path, encoding="utf-8"):
        p = ln.split()
        if len(p) == 6:
            run.setdefault(p[0], {})[p[2]] = float(p[4])
    return run

def write_run(path, run, tag):
    """Сначала во временный файл, потом переименование: оборванная запись не
    оставляет полуфайл, который перезапуск принял бы за готовый."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path + ".tmp", "w", encoding="utf-8") as f:
        for q, docs in run.items():
            for r, (d, s) in enumerate(sorted(docs.items(), key=lambda x: -x[1]), 1):
                f.write(f"{q}\tQ0\t{d}\t{r}\t{s:.6f}\t{tag}\n")
    os.replace(path + ".tmp", path)

In [ ]:
# ── 5. LLM (грузится до Java: vLLM запускает свой процесс, JVM ему не нужна) ──
from vllm import SamplingParams
if LLM_MODEL == "auto":
    LLM_MODEL = "Qwen/Qwen3-8B" if BIG else "Qwen/Qwen3-4B-Instruct-2507"
llm = load_llm(LLM_MODEL, MAXLEN)
tok = llm.get_tokenizer()
SP = SamplingParams(temperature=0.0, max_tokens=MAX_NEW)
SHORT = LLM_MODEL.split("/")[-1]
QGEN = f"{DRIVE}/qgen/{SHORT}"
os.makedirs(QGEN, exist_ok=True)
SYSTEM = "You help a search engine find documents that answer the latest question in a conversation."

def ask(kind, split, items):
    """items: [(ключ, промпт)] → {ключ: ответ}. Кэш на Drive: qgen/<модель>/<kind>_<split>.jsonl."""
    path = f"{QGEN}/{kind}_{split}.jsonl"
    got = {}
    if os.path.exists(path):
        got = {tuple(r["key"]): r["out"] for r in map(json.loads, open(path, encoding="utf-8"))}
    todo = [(k, p) for k, p in items if k not in got]
    if todo:
        t0 = time.time()
        outs = llm.generate([chat(tok, [{"role": "system", "content": SYSTEM}, {"role": "user", "content": p}])
                             for _, p in todo], SP, use_tqdm=False)
        with open(path, "a", encoding="utf-8") as f:
            for (k, _), o in zip(todo, outs):
                got[k] = o.outputs[0].text.strip()
                f.write(json.dumps({"key": list(k), "out": got[k]}, ensure_ascii=False) + "\n")
        print(f"  LLM {kind} {split}: {len(todo)} за {(time.time() - t0) / 60:.1f} мин")
    return got
print("LLM:", LLM_MODEL)

In [ ]:
# ── 6. BM25 кита + проверка: BM25 + история на dev = таблица организаторов ──
from official_baseline import OfficialBM25
from pyserini.analysis import Analyzer, get_lucene_analyzer
AN = Analyzer(get_lucene_analyzer())
_BM = {}

def bm25(dom):
    if dom not in _BM:
        ids, texts = load_corpus(dom)
        _BM[dom] = OfficialBM25(texts, ids)
    return _BM[dom]

def search(dom, qids, texts):
    """То же, что OfficialBM25.search, но топ берётся numpy (устойчивая сортировка —
    равные баллы в порядке корпуса, как в ките) и сразу обрезается до TOPK."""
    b = bm25(dom)
    ids = b.doc_ids
    run = {}
    for q, t in zip(qids, texts):
        sims = np.asarray(b.index[b.model[b.dictionary.doc2bow(b.analyzer.analyze(t))]])
        top = np.argsort(-sims, kind="stable")[:TOPK]
        run[q] = {ids[i]: float(sims[i]) for i in top}
    return run

t0 = time.time()
for dom in DOMAINS:
    qids, qtexts = topics(dom, "dev", "hist")
    got = ndcg10(search(dom, qids, qtexts), dom, "dev")
    print(f"  {dom:<20} {got:.4f}  таблица {BM25_DEV[dom]:.4f}  · {(time.time() - t0) / 60:.1f} мин")
    assert abs(got - BM25_DEV[dom]) < 0.0015, "BM25 не совпал с официальным — дальше не считать"
print("BM25 совпадает с официальным")

In [ ]:
# ── 7. Реплики и промпты ─────────────────────────────────────────────────────
NO_HIST = "No previous conversation."

def load_turns(split):
    """Те же реплики, что topics(): только с золотом. h — conversation_history как есть,
    prev — прошлые пары (вопрос, ответ) диалога. Золото и ответ текущей реплики не берутся."""
    out = []
    for dom in DOMAINS:
        for c in json.load(open(f"{DATA}/{dom}/benchmark_{split}.json", encoding="utf-8")):
            prev = []
            for t in c["turns"]:
                if t.get("gold_doc_ids") or t.get("supporting_doc_ids"):
                    h = t.get("conversation_history", "")
                    out.append({"key": (dom, f"{c['id']}_turn_{t['turn_id']}"), "q": t["query"],
                                "h": "" if h == NO_HIST else h, "prev": list(prev)})
                prev.append((t["query"], t["answer"]))
    return out

def with_hist(q, h):
    return q + (f"\n\nConversation History:\n{h}" if h else "")

def pairs(prev):
    return "\n".join(f"Q: {q}\nA: {a}" for q, a in prev)

def conv(t):
    return f"Conversation so far:\n{t['h'] or '(none)'}\n\nLatest question: {t['q']}"

P_STEPBACK = ("{c}\n\nWrite one more general step-back question about the underlying concept or principle "
              "that must be understood to answer the latest question. Output only that question.")
P_SUBQ = ("{c}\n\nBreak the latest question into 2-3 simple sub-questions that are self-contained search "
          "queries (resolve every reference to the conversation). Output one per line, nothing else.")
P_RE = ("In a conversation, the user asked: {q}\n\nThe assistant answered:\n{a}\n\nRewrite this last answer so "
        "it is fully self-contained: replace pronouns and references with what they mean. Output only the rewritten answer.")
P_PASSAGE = ("{c}\n\nWrite a short encyclopedic passage (60-100 words) that answers the latest question. "
             "Output only the passage.")
P_REFINE = ("{c}\n\nA search engine returned these passages; some may be irrelevant:\n{docs}\n\nUsing the useful "
            "information in them and your own knowledge, write a correct encyclopedic passage (60-100 words) that "
            "answers the latest question. Output only the passage.")

def keywords(passage, base, n=N_KW):
    """Слова абзаца, чьих основ (анализатор Lucene) нет в запросе; топ-n по частоте,
    при равенстве — по первому появлению."""
    have = set(AN.analyze(base))
    cnt, form = {}, {}
    for w in re.findall(r"[A-Za-z][A-Za-z0-9-]+", passage):
        st = AN.analyze(w)
        if len(st) != 1 or st[0] in have:
            continue
        cnt[st[0]] = cnt.get(st[0], 0) + 1
        form.setdefault(st[0], w.lower())
    return " ".join(form[s] for s in sorted(cnt, key=lambda s: -cnt[s])[:n])

def rrf(runs, k=60):
    s = {}
    for run in runs:
        for r, (d, _) in enumerate(sorted(run.items(), key=lambda x: -x[1]), 1):
            s[d] = s.get(d, 0) + 1 / (k + r)
    return dict(sorted(s.items(), key=lambda x: -x[1])[:TOPK])

In [ ]:
# ── 8. Варианты: тексты → Drive/reteco/queries, BM25 → Drive/reteco/runs ────
TEXTS = {}      # (вариант, split) → {ключ: [тексты]}; несколько текстов сливаются RRF

def save_variant(name, split, texts):
    """texts: {ключ: [текст, ...]}. В queries пишется один текст (склейка) — для плотных
    ноутбуков; BM25 ищет каждым текстом отдельно и сливает RRF."""
    TEXTS[(name, split)] = texts
    d = f"{QUERIES}/{name}"
    os.makedirs(d, exist_ok=True)
    with open(f"{d}/{split}.jsonl.tmp", "w", encoding="utf-8") as f:
        for (dom, qid), ts in texts.items():
            f.write(json.dumps({"domain": dom, "topic_id": qid, "text": "\n\n".join(ts)}, ensure_ascii=False) + "\n")
    os.replace(f"{d}/{split}.jsonl.tmp", f"{d}/{split}.jsonl")
    json.dump({"llm": LLM_MODEL, "notebook": "query.ipynb"}, open(f"{d}/config.json", "w"))

def run_variant(name, split):
    rd = f"{RUNS}/bm25_{name}_{split}"
    texts = TEXTS[(name, split)]
    for dom in DOMAINS:
        path = f"{rd}/{dom}/run.trec"
        if os.path.exists(path):
            continue
        items = [(qid, ts) for (d, qid), ts in texts.items() if d == dom]
        n = max(len(ts) for _, ts in items)
        runs = [search(dom, [q for q, ts in items if len(ts) > i], [ts[i] for _, ts in items if len(ts) > i])
                for i in range(n)]
        run = {q: (rrf([r[q] for r in runs if q in r]) if len(ts) > 1 else runs[0][q]) for q, ts in items}
        write_run(path, run, f"bm25_{name}")
    json.dump({"variant": name, "split": split, "llm": LLM_MODEL, "bm25": "kit OfficialBM25 k1=0.9 b=0.4",
               "topk": TOPK, "date": time.strftime("%Y-%m-%dT%H:%M:%S")}, open(f"{rd}/config.json", "w"), indent=2)
    return rd

def top_texts(rd, split, turns, k=FEEDBACK_K):
    """Тексты топ-k прогона для каждой реплики — это видит LLM в QExpand (без doc_id)."""
    out = {}
    for dom in DOMAINS:
        ids, docs = load_corpus(dom)
        text = dict(zip(ids, docs))
        run = read_run(f"{rd}/{dom}/run.trec")
        for t in turns:
            if t["key"][0] == dom:
                top = sorted(run.get(t["key"][1], {}).items(), key=lambda x: -x[1])[:k]
                out[t["key"]] = [" ".join(text[d].split()[:DOC_WORDS]) for d, _ in top]
    return out

def refine_prompts(turns, tops):
    return [(t["key"], P_REFINE.format(c=conv(t), docs="\n\n".join(f"[{i}] {d}" for i, d in enumerate(tops[t["key"]], 1))))
            for t in turns]

for split in SPLITS:
    turns = load_turns(split)
    base = {t["key"]: with_hist(t["q"], t["h"]) for t in turns}
    later = [t for t in turns if t["prev"]]
    sb = ask("stepback", split, [(t["key"], P_STEPBACK.format(c=conv(t))) for t in turns])
    sq = ask("subq", split, [(t["key"], P_SUBQ.format(c=conv(t))) for t in turns])
    re_ = ask("re", split, [(t["key"], P_RE.format(q=t["prev"][-1][0], a=t["prev"][-1][1])) for t in later])
    ps = ask("passage", split, [(t["key"], P_PASSAGE.format(c=conv(t))) for t in turns])
    subqs = lambda k: [re.sub(r"^\s*(\d+[.)]|[-*•])\s*", "", l).strip() for l in sq[k].splitlines() if l.strip()][:3]
    save_variant("win1", split, {t["key"]: [with_hist(t["q"], pairs(t["prev"][-1:]))] for t in turns})
    save_variant("win2", split, {t["key"]: [with_hist(t["q"], pairs(t["prev"][-2:]))] for t in turns})
    save_variant("stepback", split, {k: [f"{b}\n\n{sb[k]}"] for k, b in base.items()})
    save_variant("subq", split, {k: [b] + subqs(k) for k, b in base.items()})
    save_variant("re", split, {t["key"]: [with_hist(t["q"], pairs(t["prev"][:-1] + [(t["prev"][-1][0], re_[t["key"]])]))
                                          if t["prev"] else t["q"]] for t in turns})
    save_variant("q2d_open", split, {k: [f"{b}\n\n{ps[k]}"] for k, b in base.items()})
    save_variant("kw", split, {k: [f"{b}\n\n{keywords(ps[k], b)}"] for k, b in base.items()})
    for name in ["win1", "win2", "stepback", "subq", "re", "q2d_open", "kw"]:
        run_variant(name, split)
    print(f"{split}: разовые варианты готовы")
    # QExpand: абзац → поиск → LLM видит топ-5 и переписывает → поиск → ещё раз
    prev_run = f"{RUNS}/bm25_q2d_open_{split}"
    for r in (1, 2):
        ex = ask(f"refine{r}", split, refine_prompts(turns, top_texts(prev_run, split, turns)))
        save_variant(f"qexp{r}", split, {k: [f"{b}\n\n{ex[k]}"] for k, b in base.items()})
        prev_run = run_variant(f"qexp{r}", split)
    print(f"{split}: QExpand готов")

In [ ]:
# ── 9. Итог: nDCG@10 макро по доменам ───────────────────────────────────────
REF = {"bm25_hist": BM25_MACRO, "bm25_q2d (Claude)": {"dev": 0.5463, "train": 0.5584}}
names = ["win1", "win2", "stepback", "subq", "re", "q2d_open", "kw", "qexp1", "qexp2"]
print(f"{'вариант':<22}" + "".join(f"{s:>9}" for s in SPLITS))
for n, v in REF.items():
    print(f"{n:<22}" + "".join(f"{v[s]:>9.4f}" for s in SPLITS))
for n in names:
    row = []
    for s in SPLITS:
        rd = f"{RUNS}/bm25_{n}_{s}"
        ok = all(os.path.exists(f"{rd}/{d}/run.trec") for d in DOMAINS)
        row.append(sum(ndcg10(read_run(f"{rd}/{d}/run.trec"), d, s) for d in DOMAINS) / len(DOMAINS) if ok else float("nan"))
    print(f"{n:<22}" + "".join(f"{x:>9.4f}" for x in row))
print("\nВыбирать по train. Дальше: скачать", RUNS, "и", QUERIES, "в проект, сказать Claude.")